In [ ]:
"""
==============================================================================
RETAIL CREDIT PROJECT (BRAZIL) - STAGE 3.1: EXPECTED COST CURVE AND CUT-OFF
Decision Calibration via Cost Matrix with Calibrated Probabilities
Update: Multi-Algorithm Cross-Validation (Random Forest vs CatBoost) 
and Financial Bootstrap.
Author: Itaiguara de Oliveira Bezerra
==============================================================================
"""

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.utils import resample
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["grid.color"] = "#dcdcdc"
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.5

def remove_chart_spines(ax=None):
    if ax is None: ax = plt.gca()
    for spine in ["top", "right", "left", "bottom"]: ax.spines[spine].set_visible(False)

# NOTE: Original Portuguese strings kept here to match the raw .parquet database
MODALIDADES_EXPURGO = [
    "Financiamentos à exportação", "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários", "Financiamentos com interveniência"
]

def map_macro_block_vectorized(df: pd.DataFrame) -> np.ndarray:
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()
    cond_rotativo = sub.str.contains("crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|antecipação de fatura", regex=True, na=False)
    cond_consignado = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_garantia_real = (sub.str.contains("habitacional|sfh|veículos automotores|veículos autom.|home equity|veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False) | mod.str.contains("imobiliários", regex=True, na=False))
    
    # Translated Macroblock Names
    choices = ["1. Revolving / Unsecured", "2. Payroll / Secured", "3. Real Collateral (Mortgage/Auto)"]
    conditions = [cond_rotativo, cond_consignado, cond_garantia_real]
    
    return np.select(conditions, choices, default="4. Others / Renegotiated")

def load_local_csv(file_name, col_name):
    if not os.path.exists(f"{file_name}.csv"):
        raise RuntimeError(f"Error: File '{file_name}.csv' not found.")
    df = pd.read_csv(f"{file_name}.csv", sep=";", decimal=",", encoding="latin-1")
    # Dictionary mapping remains in PT to correctly parse the raw CSV data
    meses_pt = {
        'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
        'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
    }
    df.columns = ['data', 'valor']
    df = df[df['data'].astype(str).str.contains(r'/', na=False)].copy()
    df['data'] = df['data'].str.lower()
    for mes_txt, mes_num in meses_pt.items(): df['data'] = df['data'].str.replace(f"{mes_txt}/", f"{mes_num}/")
    
    def adjust_year(data_str):
        partes = str(data_str).split('/')
        if len(partes) == 2 and len(partes[1]) == 2:
            ano = int(partes[1])
            partes[1] = f"19{ano}" if ano > 50 else f"20{ano:02d}"
        return "/".join(partes)
    
    df['data'] = df['data'].apply(adjust_year)
    df['data'] = pd.to_datetime(df['data'], format='%m/%Y', errors='coerce')
    if df['valor'].dtype == 'O': df['valor'] = df['valor'].str.replace(',', '.')
    df[col_name] = pd.to_numeric(df['valor'], errors='coerce')
    df['ano_mes'] = df['data'].dt.to_period('M')
    return df[['ano_mes', col_name]].dropna().drop_duplicates(subset=['ano_mes'])

def execute_stage3_1_economic_cutoff():
    print("=" * 95)
    print("RETAIL CREDIT PROJECT - STAGE 3.1: EXPECTED COST CURVE AND OPTIMAL CUT-OFF")
    print("=" * 95)

    print(">>> [1/6] Loading macroeconomic series via local CSVs...")
    df_selic = load_local_csv("selic", "selic_media")
    df_juros = load_local_csv("juros_pf", "juros_pf")
    df_endiv = load_local_csv("endividamento_familias", "endividamento_familias")

    df_macro = pd.merge(df_selic, df_juros, on="ano_mes", how="inner")
    df_macro = pd.merge(df_macro, df_endiv, on="ano_mes", how="inner").sort_values("ano_mes").reset_index(drop=True)
    df_macro["spread_bancario"] = df_macro["juros_pf"] - df_macro["selic_media"]
    df_macro["diff_selic_12m"] = df_macro["selic_media"].diff(12)
    df_macro["diff_endiv_12m"] = df_macro["endividamento_familias"].diff(12)
    df_macro["selic_lag3"] = df_macro["selic_media"].shift(3)
    df_macro["selic_lag6"] = df_macro["selic_media"].shift(6)
    df_macro["endiv_lag6"] = df_macro["endividamento_familias"].shift(6)

    print(">>> [2/6] Reading SCR in batches for RAM protection...")
    cols_load = ["data_base", "modalidade", "submodalidade", "cnae_ocupacao", "porte", "carteira_ativa", "vencido_acima_de_90_dias", "numero_de_operacoes"]
    parquet_file = pq.ParquetFile("painel_scr_pf_bruto.parquet")
    lista_paineis = []

    for batch in parquet_file.iter_batches(columns=cols_load, batch_size=500_000):
        df_chunk = batch.to_pandas()
        df_chunk = df_chunk[~df_chunk["modalidade"].isin(MODALIDADES_EXPURGO)]
        if len(df_chunk) == 0: continue
        df_chunk["macrobloco"] = map_macro_block_vectorized(df_chunk)
        df_chunk["data_base"] = pd.to_datetime(df_chunk["data_base"])
        df_parcial = df_chunk.groupby(["data_base", "macrobloco", "cnae_ocupacao", "porte"]).agg(
            carteira_ativa=("carteira_ativa", "sum"), vencido_acima_de_90_dias=("vencido_acima_de_90_dias", "sum"), numero_de_operacoes=("numero_de_operacoes", "sum")
        ).reset_index()
        lista_paineis.append(df_parcial)

    df_painel_bruto = pd.concat(lista_paineis, ignore_index=True)
    df_painel = df_painel_bruto.groupby(["data_base", "macrobloco", "cnae_ocupacao", "porte"]).agg(
        carteira_ativa=("carteira_ativa", "sum"), vencido_acima_de_90_dias=("vencido_acima_de_90_dias", "sum"), numero_de_operacoes=("numero_de_operacoes", "sum")
    ).reset_index()

    df_painel["taxa_inadimplencia_pct"] = np.where(df_painel["carteira_ativa"] > 0, (df_painel["vencido_acima_de_90_dias"] / df_painel["carteira_ativa"]) * 100, 0.0)
    df_painel["ano_mes"] = df_painel["data_base"].dt.to_period("M")
    df_painel = df_painel.sort_values(["macrobloco", "cnae_ocupacao", "porte", "data_base"]).reset_index(drop=True)

    ticket_bruto = np.where(df_painel["numero_de_operacoes"] > 0, (df_painel["carteira_ativa"] / df_painel["numero_de_operacoes"]) / 1000.0, 1.0)
    df_painel["ticket_medio_mil"] = ticket_bruto
    media_mb = df_painel.groupby("macrobloco")["ticket_medio_mil"].transform("mean")
    df_painel["log_ticket_medio_rel"] = np.log1p(df_painel["ticket_medio_mil"] / np.maximum(media_mb, 0.01))

    df_painel["crescimento_carteira_interanual"] = (df_painel.groupby(["macrobloco", "cnae_ocupacao", "porte"])["carteira_ativa"].pct_change(12) * 100).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-100, 500)
    df_painel["crescimento_operacoes_interanual"] = (df_painel.groupby(["macrobloco", "cnae_ocupacao", "porte"])["numero_de_operacoes"].pct_change(12) * 100).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-100, 500)

    HORIZONTE = 12
    df_painel["taxa_inad_futura"] = df_painel.groupby(["macrobloco", "cnae_ocupacao", "porte"])["taxa_inadimplencia_pct"].shift(-HORIZONTE)
    df_final = pd.merge(df_painel, df_macro, on="ano_mes", how="inner").dropna(subset=["taxa_inad_futura"]).reset_index(drop=True)

    corte_val = pd.to_datetime("2020-01-01")
    corte_oot = pd.to_datetime("2022-01-01")

    df_train = df_final[df_final["data_base"] < corte_val].copy()
    df_val = df_final[(df_final["data_base"] >= corte_val) & (df_final["data_base"] < corte_oot)].copy()
    df_oot = df_final[df_final["data_base"] >= corte_oot].copy()

    tau_treino = df_train.groupby("macrobloco").apply(lambda g: (g["vencido_acima_de_90_dias"].sum() / g["carteira_ativa"].sum()) * 100).to_dict()

    for d_sub in [df_train, df_val, df_oot]:
        d_sub["tau_treino"] = d_sub["macrobloco"].map(tau_treino)
        d_sub["alvo_inadimplencia_alta"] = (d_sub["taxa_inad_futura"] > d_sub["tau_treino"]).astype(int)

    cat_features = ["macrobloco", "cnae_ocupacao", "porte"]
    num_features = ["diff_selic_12m", "selic_lag3", "selic_lag6", "spread_bancario", "diff_endiv_12m", "endiv_lag6", "log_ticket_medio_rel", "crescimento_carteira_interanual", "crescimento_operacoes_interanual"]
    all_features = cat_features + num_features
    
    # Dummy preparation for Random Forest
    df_train_enc = pd.get_dummies(df_train[all_features], columns=cat_features, dtype=float)
    df_val_enc = pd.get_dummies(df_val[all_features], columns=cat_features, dtype=float)
    df_oot_enc = pd.get_dummies(df_oot[all_features], columns=cat_features, dtype=float)
    df_train_enc, df_val_enc = df_train_enc.align(df_val_enc, join="left", axis=1, fill_value=0.0)
    df_train_enc, df_oot_enc = df_train_enc.align(df_oot_enc, join="left", axis=1, fill_value=0.0)

    print(">>> [3/6] Training CatBoost and Random Forest (Multi-Algorithm)...")
    
    # Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)
    rf.fit(df_train_enc, df_train["alvo_inadimplencia_alta"])
    prob_val_rf = rf.predict_proba(df_val_enc)[:, 1]
    prob_oot_rf = rf.predict_proba(df_oot_enc)[:, 1]
    
    # CatBoost (Isotonically Calibrated)
    pool_tr = Pool(df_train[all_features], df_train["alvo_inadimplencia_alta"], cat_features=cat_features)
    pool_va = Pool(df_val[all_features], df_val["alvo_inadimplencia_alta"], cat_features=cat_features)
    cb_base = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6, random_seed=42, verbose=False)
    cb_base.fit(pool_tr, eval_set=pool_va, early_stopping_rounds=40, verbose=False)
    calibrador = CalibratedClassifierCV(cb_base, cv="prefit", method="isotonic")
    calibrador.fit(df_val[all_features], df_val["alvo_inadimplencia_alta"])
    prob_val_cb = calibrador.predict_proba(df_val[all_features])[:, 1]
    prob_oot_cb = calibrador.predict_proba(df_oot[all_features])[:, 1]
    
    y_val = df_val["alvo_inadimplencia_alta"].values
    y_oot = df_oot["alvo_inadimplencia_alta"].values

    print("\n>>> [4/6] Optimizing Economic Cut-off STRICTLY on Validation (2020-2021)...")
    C_FN = 0.65  # Effective loss cost (LGD) over defaulted concession
    C_FP = 0.18  # Opportunity cost of net margin

    cutoffs = np.linspace(0.01, 0.99, 99)
    custos_val_rf, custos_val_cb = [], []

    for c in cutoffs:
        cm_rf = confusion_matrix(y_val, (prob_val_rf >= c).astype(int), labels=[0, 1])
        custos_val_rf.append((C_FN * cm_rf[1,0] + C_FP * cm_rf[0,1]) / len(y_val))
        
        cm_cb = confusion_matrix(y_val, (prob_val_cb >= c).astype(int), labels=[0, 1])
        custos_val_cb.append((C_FN * cm_cb[1,0] + C_FP * cm_cb[0,1]) / len(y_val))

    c_otimo_rf = cutoffs[np.argmin(custos_val_rf)]
    c_otimo_cb = cutoffs[np.argmin(custos_val_cb)]
    c_teorico = C_FP / (C_FN + C_FP)

    print(f"  * Theoretical Bayesian Cut-off: {c_teorico:.3f}")
    print(f"  * Calibrated Random Forest Cut-off: {c_otimo_rf:.3f}")
    print(f"  * Calibrated CatBoost Cut-off: {c_otimo_cb:.3f}")

    print("\n>>> [5/6] Applying Frozen Decision on OOT Test (2022-2026)...")
    saldo_medio_linha = df_oot["carteira_ativa"].mean()
    custos_oot_cb, fn_list_oot, fp_list_oot, perda_monetaria_oot = [], [], [], []

    for c in cutoffs:
        cm = confusion_matrix(y_oot, (prob_oot_cb >= c).astype(int), labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        custos_oot_cb.append((C_FN * fn + C_FP * fp) / len(y_oot))
        perda_r_milhoes = ((C_FN * fn * saldo_medio_linha) + (C_FP * fp * saldo_medio_linha)) / 1e6
        perda_monetaria_oot.append(perda_r_milhoes)
        fn_list_oot.append(fn)
        fp_list_oot.append(fp)

    idx_otimo = np.argmin(np.abs(cutoffs - c_otimo_cb))
    custo_otimo_oot = custos_oot_cb[idx_otimo]
    perda_otimo_oot = perda_monetaria_oot[idx_otimo]

    idx_50 = np.argmin(np.abs(cutoffs - 0.50))
    custo_50 = custos_oot_cb[idx_50]
    perda_50 = perda_monetaria_oot[idx_50]
    
    idx_teo = np.argmin(np.abs(cutoffs - c_teorico))
    perda_teorico = perda_monetaria_oot[idx_teo]

    economia_emp_pct = ((custo_50 - custo_otimo_oot) / custo_50) * 100
    economia_emp_r = perda_50 - perda_otimo_oot

    # ----------------------------------------------------------------------------
    # STATISTICAL INFERENCE: MCNEMAR'S TEST AND COST BOOTSTRAP
    # ----------------------------------------------------------------------------
    print("\n>>> [6/6] Statistical Inference: McNemar's Test and Cost Bootstrap...")
    pred_oot_padrao = (prob_oot_cb >= 0.50).astype(int)
    pred_oot_bayes = (prob_oot_cb >= c_teorico).astype(int)
    pred_oot_otimo = (prob_oot_cb >= c_otimo_cb).astype(int)

    acertos_bayes = (pred_oot_bayes == y_oot).astype(int)
    acertos_otimo = (pred_oot_otimo == y_oot).astype(int)
    tabela_mcnemar = confusion_matrix(acertos_bayes, acertos_otimo)
    p_valor_mcnemar = mcnemar(tabela_mcnemar, exact=False, correction=True).pvalue

    np.random.seed(42)
    indices_oot = np.arange(len(y_oot))
    economia_boot_r = []
    
    for i in range(1000):
        idx_boot = resample(indices_oot, random_state=i)
        y_b = y_oot[idx_boot]
        cm_50 = confusion_matrix(y_b, pred_oot_padrao[idx_boot], labels=[0, 1]).ravel()
        perda_50_b = ((C_FN * cm_50[2] * saldo_medio_linha) + (C_FP * cm_50[1] * saldo_medio_linha)) / 1e6
        cm_ot = confusion_matrix(y_b, pred_oot_otimo[idx_boot], labels=[0, 1]).ravel()
        perda_ot_b = ((C_FN * cm_ot[2] * saldo_medio_linha) + (C_FP * cm_ot[1] * saldo_medio_linha)) / 1e6
        economia_boot_r.append(perda_50_b - perda_ot_b)

    ic_inf = np.percentile(economia_boot_r, 2.5)
    ic_sup = np.percentile(economia_boot_r, 97.5)

    # ----------------------------------------------------------------------------
    # EDITORIAL CHARTS
    # ----------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=300)

    ax1 = axes[0]
    ax1.plot(cutoffs, custos_oot_cb, color="#2ca02c", lw=2.5, label="Expected Cost (CatBoost OOT)")
    ax1.axvline(c_teorico, color="#9467bd", linestyle="-.", lw=1.6, label=f"Bayesian Cut-off c* = {c_teorico:.3f}")
    ax1.axvline(c_otimo_cb, color="#2ca02c", linestyle="--", lw=1.8, label=f"Calibrated Cut-off c* = {c_otimo_cb:.2f}")
    ax1.axvline(0.50, color="#d62728", linestyle=":", lw=1.8, label="Default Cut-off c = 0.50")
    
    if c_otimo_cb <= 0.50:
        ax1.fill_between(cutoffs, custos_oot_cb, custo_50, where=(cutoffs <= 0.50) & (cutoffs >= c_otimo_cb), color="#2ca02c", alpha=0.15, label=f"Efficiency Zone (Savings: {economia_emp_pct:.1f}%)")

    ax1.set_title("1. Credit Decision Expected Cost Curve", fontsize=11.5, fontweight="bold", pad=12)
    ax1.set_xlabel("Decision Cut-off (c)", fontsize=10, fontweight="bold")
    ax1.set_ylabel("Average Expected Cost per Operation", fontsize=10, fontweight="bold")
    ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=9)
    ax1.grid(True); remove_chart_spines(ax1)

    ax2 = axes[1]
    ax2.plot(cutoffs, fn_list_oot, color="#d62728", lw=2.2, label="False Negatives (LGD Risk)")
    ax2.plot(cutoffs, fp_list_oot, color="#ff7f0e", lw=2.2, label="False Positives (Opportunity Cost)")
    ax2.axvline(c_teorico, color="#9467bd", linestyle="-.", lw=1.6, label=f"Bayesian Cut-off c* = {c_teorico:.3f}")
    ax2.axvline(c_otimo_cb, color="#2ca02c", linestyle="--", lw=1.8, label=f"Calibrated Cut-off c* = {c_otimo_cb:.2f}")
    ax2.axvline(0.50, color="#d62728", linestyle=":", lw=1.8, label="Default Cut-off c = 0.50")

    ax2.set_title("2. Operational Balance between Type I and Type II Errors", fontsize=11.5, fontweight="bold", pad=12)
    ax2.set_xlabel("Decision Cut-off (c)", fontsize=10, fontweight="bold")
    ax2.set_ylabel("Error Quantity (OOT)", fontsize=10, fontweight="bold")
    ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=9)
    ax2.grid(True); remove_chart_spines(ax2)

    plt.subplots_adjust(bottom=0.22)
    nome_fig = "panel_stage3_1_expected_cost_curve_en.png"
    plt.savefig(nome_fig, dpi=300, bbox_inches="tight")
    plt.close()

    # ----------------------------------------------------------------------------
    # APPENDIX: COMPARATIVE CONFUSION MATRICES (c=0.50 vs c=Optimal)
    # ----------------------------------------------------------------------------
    fig_cm, axes_cm = plt.subplots(1, 2, figsize=(14, 6), dpi=300)
    
    # Matrix for Cut-off = 0.50
    cm_padrao = confusion_matrix(y_oot, pred_oot_padrao)
    sns.heatmap(cm_padrao, annot=True, fmt='d', cmap="Reds", cbar=False,
                xticklabels=["Low Risk (0)", "High Risk (1)"],
                yticklabels=["Low Risk (0)", "High Risk (1)"],
                ax=axes_cm[0], annot_kws={"size": 16, "weight": "bold"})
    axes_cm[0].set_title("CatBoost: Default Cut-off (c = 0.50)", fontsize=14, fontweight="bold", pad=15)
    axes_cm[0].set_xlabel("Model Prediction", fontsize=12, fontweight="bold")
    axes_cm[0].set_ylabel("Reality (OOT)", fontsize=12, fontweight="bold")
    
    # Matrix for Cut-off = c_otimo_cb
    cm_otimo = confusion_matrix(y_oot, pred_oot_otimo)
    sns.heatmap(cm_otimo, annot=True, fmt='d', cmap="Greens", cbar=False,
                xticklabels=["Low Risk (0)", "High Risk (1)"],
                yticklabels=["Low Risk (0)", "High Risk (1)"],
                ax=axes_cm[1], annot_kws={"size": 16, "weight": "bold"})
    axes_cm[1].set_title(f"CatBoost: Optimal Economic Cut-off (c = {c_otimo_cb:.2f})", fontsize=14, fontweight="bold", pad=15)
    axes_cm[1].set_xlabel("Model Prediction", fontsize=12, fontweight="bold")
    axes_cm[1].set_ylabel("Reality (OOT)", fontsize=12, fontweight="bold")
    
    plt.tight_layout()
    nome_fig_cm = "appendix_confusion_matrices_cutoff_en.png"
    plt.savefig(nome_fig_cm, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  -> Appendix saved: '{nome_fig_cm}'")

    # ----------------------------------------------------------------------------
    # RECONCILED SUMMARY TABLE
    # ----------------------------------------------------------------------------
    tn_p, fp_p, fn_p, tp_p = confusion_matrix(y_oot, pred_oot_padrao, labels=[0, 1]).ravel()
    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_oot, pred_oot_bayes, labels=[0, 1]).ravel()
    tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_oot, pred_oot_otimo, labels=[0, 1]).ravel()

    df_resumo = pd.DataFrame([
        {
            "Decision Regime": "Default Cut-off (c = 0.50)", "Cut-off": 0.50,
            "Sensitivity": f"{(tp_p / (tp_p + fn_p)):.2%}", "Specificity": f"{(tn_p / (tn_p + fp_p)):.2%}",
            "False Negatives (LGD)": fn_p, "False Positives (Opportunity)": fp_p,
            "Simulated Total Loss (BRL Mn)": round(perda_50, 2),
        },
        {
            "Decision Regime": f"Theoretical Bayesian (c = {c_teorico:.2f})", "Cut-off": round(c_teorico, 2),
            "Sensitivity": f"{(tp_b / (tp_b + fn_b)):.2%}", "Specificity": f"{(tn_b / (tn_b + fp_b)):.2%}",
            "False Negatives (LGD)": fn_b, "False Positives (Opportunity)": fp_b,
            "Simulated Total Loss (BRL Mn)": round(perda_teorico, 2),
        },
        {
            "Decision Regime": f"Calibrated CatBoost (c = {c_otimo_cb:.2f})", "Cut-off": round(c_otimo_cb, 2),
            "Sensitivity": f"{(tp_o / (tp_o + fn_o)):.2%}", "Specificity": f"{(tn_o / (tn_o + fp_o)):.2%}",
            "False Negatives (LGD)": fn_o, "False Positives (Opportunity)": fp_o,
            "Simulated Total Loss (BRL Mn)": round(perda_otimo_oot, 2),
        },
    ])

    nome_excel = "report_stage3_1_retail_economic_cutoff_en.xlsx"
    with pd.ExcelWriter(nome_excel, engine="openpyxl") as writer: df_resumo.to_excel(writer, sheet_name="Cutoff_Decision_Synthesis", index=False)

    print("\n" + "=" * 95)
    print("ECONOMIC SYNTHESIS OF CALIBRATION (OOT 2022-2026):")
    print("=" * 95)
    print(df_resumo.to_string(index=False))
    print("-" * 95)
    print(f"--> Estimated Point Simulated Savings: BRL {economia_emp_r:.2f} Million")
    print(f"--> Financial Confidence Interval (95% Bootstrap): [BRL {ic_inf:.2f} Mn, BRL {ic_sup:.2f} Mn]")
    print(f"--> P-Value (McNemar Bayes vs Optimal): {p_valor_mcnemar:.4e}")
    print("=" * 95)

if __name__ == "__main__":
    execute_stage3_1_economic_cutoff()